In [ ]:
!pip install transformers torch kagglehub scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import kagglehub

path = kagglehub.dataset_download("hongtrung/clinc150-dataset")
print(path)

100%|██████████| 1.02M/1.02M [00:00<00:00, 128MB/s]

Extracting files...
/root/.cache/kagglehub/datasets/hongtrung/clinc150-dataset/versions/1


In [ ]:
import json
import os

file_path = os.path.join(path, "data", "data_full.json")

with open(file_path) as f:
    data = json.load(f)

texts = []
labels = []


for split in ['train', 'val', 'test']:
    for sentence, intent in data[split]:
        texts.append(sentence)
        labels.append(intent)

print("Total samples:", len(texts))

Total samples: 22500


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
labels = le.fit_transform(labels)
num_labels = len(set(labels))

print("Total intents:", num_labels)

Total intents: 150


In [ ]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2
)

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
import torch

class ChatDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import Trainer, TrainingArguments

train_dataset = ChatDataset(train_encodings, train_labels)
val_dataset = ChatDataset(val_encodings, val_labels)

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/chatbot_model",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    # evaluation_strategy="epoch", # Removed to solve TypeError
    # save_strategy="epoch",       # Removed to solve TypeError
    logging_dir="./logs"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Step,Training Loss
500,3.822720
1000,1.589550
1500,0.639966
2000,0.312480
2500,0.152518
3000,0.113856
3500,0.069298
4000,0.083940
4500,0.092641
5000,0.042150


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6750, training_loss=0.5198258130815294, metrics={'train_runtime': 701.2984, 'train_samples_per_second': 77.0, 'train_steps_per_second': 9.625, 'total_flos': 861392945448000.0, 'train_loss': 0.5198258130815294, 'epoch': 3.0})

In [ ]:
model_path = "/content/drive/MyDrive/chatbot_model"

model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

# Save label encoder
import pickle
with open(model_path + "/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

print("Model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!


In [ ]:
from sklearn.metrics import accuracy_score
import torch

# Determine the device to use (GPU if available, else CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval() # Set the model to evaluation mode

preds = []
true_labels = []

with torch.no_grad(): # Disable gradient calculation for inference
    for i in range(len(val_texts)):
        inputs = tokenizer(val_texts[i], return_tensors="pt", truncation=True, padding=True)
        # Move all input tensors to the correct device
        inputs = {k: v.to(device) for k, v in inputs.items()}

        outputs = model(**inputs)
        pred = outputs.logits.argmax().item()

        preds.append(pred)
        true_labels.append(val_labels[i])

acc = accuracy_score(true_labels, preds)
print("Validation Accuracy:", acc)

Validation Accuracy: 0.9791111111111112


In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
import pickle

model_path = "/content/drive/MyDrive/chatbot_model"

tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)

with open(model_path + "/label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

print("Model loaded successfully!")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded successfully!


In [ ]:
import torch

def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    pred = torch.argmax(outputs.logits, dim=1).item()
    return le.inverse_transform([pred])[0]

In [ ]:
print(predict("hello"))
print(predict("what is my account balance"))
print(predict("book a flight"))

greeting
balance
book_flight


In [ ]:
responses = {
    "greeting": [
        "Hello! 👋 How can I help you?",
        "Hi there! What can I do for you?",
        "Hey! Need any help?"
    ],

    "goodbye": [
        "Goodbye! Have a nice day 😊",
        "See you later!",
        "Take care!"
    ],

    "check_balance": [
        "Your account balance is ₹10,000",
        "You currently have ₹10,000 in your account"
    ],

    "transfer_money": [
        "Sure! Please provide recipient details.",
        "How much amount do you want to transfer?"
    ],

    "book_flight": [
        "Where do you want to travel?",
        "Please tell me your destination."
    ],

    "weather": [
        "It's sunny today ☀️",
        "Weather looks pleasant!"
    ],

    "fallback": [
        "Sorry, I didn’t understand that.",
        "Can you please rephrase?",
        "I’m not sure about that."
    ]
}

In [ ]:
import random

def get_response(intent):
    if intent in responses:
        return random.choice(responses[intent])
    else:
        return random.choice(responses["fallback"])

In [ ]:
def chatbot_with_response():
    context = []

    while True:
        user = input("You: ")

        if user.lower() == "exit":
            print("Bot: Goodbye! 👋")
            break

        context.append(user)

        # use last 3 messages
        combined = " ".join(context[-3:])

        intent = predict(combined)
        response = get_response(intent)

        print("Bot:", response)

In [ ]:
chatbot_with_response()

You: Bye
Bot: Take care!
You: Sorry
Bot: See you later!
You: Hii
Bot: Goodbye! Have a nice day 😊
You: exit
Bot: Goodbye! 👋


In [ ]:
chatbot_with_response()


You: Hii
Bot: Hey! Need any help?
You: Yes
Bot: Hello! 👋 How can I help you?
You: Bye
Bot: Take care!
You: ok
Bot: See you later!
You: exit
Bot: Goodbye! 👋
